In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold,GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer, r2_score, confusion_matrix
import numpy as np
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
df = pd.read_excel('/content/all_data_0710.xlsx')

# drop all rows with nan
df=df.dropna(axis=0)

In [ ]:
aaa=pd.read_excel('/content/all_data_0710.xlsx')

In [ ]:
aaa=filter_rare_values(aaa,selected_columns)
aaa.drop(aaa[aaa['accident type'] == 'Others'].index, inplace=True)


In [ ]:
copy=aaa[['事发水域路况','Cause of Accident']]

In [ ]:
selected_columns=['accident type', '发生地种类', 'season', 'wind speed level', 'dayNight',
       'fatality', '作业情况', '损伤种类', '损伤位置', 'wind direction', '能见度等级', '是否超速',
       '路况种类', 'ship type', '船体材料', '主机类型', '船长', '船宽', '设备故障', '环境恶劣']

In [ ]:
def filter_rare_values(df, selected_cols, threshold=5):
    for col in selected_columns:
        value_counts = df[col].value_counts()
        df = df[df[col].map(value_counts) >= threshold]  # Keep rows where the value appears ≥ threshold
    return df
# df=filter_rare_values(df,selected_columns)
# df.drop(df[df['accident type'] == 'Others'].index, inplace=True)

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
model = BertModel.from_pretrained("bert-base-chinese")

def get_embedding(text):
    # Tokenize the text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the mean pooling of the last hidden state for sentence-level embeddings
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

In [ ]:
Cause_Embeddings = df['Cause of Accident'].fillna("").apply(lambda x: get_embedding(x))
Traffic_Embeddings=df['事发水域路况'].fillna("").apply(lambda x: get_embedding(x))

df['Bert_cause'],df['Bert_traffic']=Cause_Embeddings,Traffic_Embeddings

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=1000)

def get_tfidf_embedding(text):
    # Vectorize the text using the fitted TF-IDF vectorizer
    embedding = vectorizer.transform([text]).toarray().squeeze()
    return embedding

vectorizer.fit(aaa['Cause of Accident'].fillna(""))
# Apply the TF-IDF embedding function to each row in the DataFrame column
Cause_Embeddings = aaa['Cause of Accident'].fillna("").apply(lambda x: get_tfidf_embedding(x))


vectorizer.fit(copy['事发水域路况'].fillna(""))
Traffic_Embeddings= copy['事发水域路况'].fillna("").apply(lambda x: get_tfidf_embedding(x))

df['Tfidf_cause'],df['Tfidf_traffic']=Cause_Embeddings,Traffic_Embeddings

In [ ]:
df = df.drop(['Cause of Accident', '事发水域路况'], axis=1)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

class TextEmbeddingClusterModel:
    def __init__(self, n_clusters=7, seed=1):
        self.n_clusters = n_clusters
        self.seed = seed

        self.kmeans_cause_bert = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_traffic_bert = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_cause_tfidf = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_traffic_tfidf = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)

    def train(self, df):
        # Expecting these columns to contain array-like vectors
        if 'Bert_cause' in df.columns:
            cause_bert_matrix = np.vstack(df['Bert_cause'].values)
            self.kmeans_cause_bert.fit(cause_bert_matrix)
        if 'Bert_traffic' in df.columns:
            traffic_bert_matrix = np.vstack(df['Bert_traffic'].values)
            self.kmeans_traffic_bert.fit(traffic_bert_matrix)

        if 'Tfidf_cause' in df.columns:
            cause_tfidf_matrix = np.vstack(df['Tfidf_cause'].values)
            self.kmeans_cause_tfidf.fit(cause_tfidf_matrix)

        if 'Tfidf_traffic' in df.columns:
            traffic_tfidf_matrix = np.vstack(df['Tfidf_traffic'].values)
            self.kmeans_traffic_tfidf.fit(traffic_tfidf_matrix)


        return self

    def predict(self, df):
        cause_bert_clusters = None
        traffic_bert_clusters = None
        cause_tfidf_clusters = None
        traffic_tfidf_clusters = None

        if 'Bert_cause' in df.columns:
            cause_bert_matrix = np.vstack(df['Bert_cause'].values)
            cause_bert_clusters = self.kmeans_cause_bert.predict(cause_bert_matrix)

        if 'Bert_traffic' in df.columns:
            traffic_bert_matrix = np.vstack(df['Bert_traffic'].values)
            traffic_bert_clusters = self.kmeans_traffic_bert.predict(traffic_bert_matrix)

        if 'Tfidf_cause' in df.columns:
            cause_tfidf_matrix = np.vstack(df['Tfidf_cause'].values)
            cause_tfidf_clusters = self.kmeans_cause_tfidf.predict(cause_tfidf_matrix)

        if 'Tfidf_traffic' in df.columns:
            traffic_tfidf_matrix = np.vstack(df['Tfidf_traffic'].values)
            traffic_tfidf_clusters = self.kmeans_traffic_tfidf.predict(traffic_tfidf_matrix)

        return cause_tfidf_clusters, traffic_tfidf_clusters, cause_bert_clusters, traffic_bert_clusters



In [ ]:
# Create target variable
# y = df["accident type"].astype(str)

# Create target to label mapping
y_encoded, y_labels = pd.factorize(y)
target_mapping = dict(zip(range(len(y_labels)), y_labels))

# Drop the original target column
# df=df.drop(["accident type"],axis=1)

NLP_features = ['Bert_cause', 'Bert_traffic', 'Tfidf_cause', 'Tfidf_traffic']
df_NLP=df[NLP_features]
df=df.drop(NLP_features,axis=1)

# One-hot encode the remaining features
df_encoded = pd.get_dummies((df), drop_first=True)

In [ ]:
df_encoded=pd.concat([df_encoded,df_NLP],axis=1)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df_encoded, y_encoded, test_size=0.3, random_state=42)

textembedding=TextEmbeddingClusterModel()
textembedding.train(X_train)
cause_tfidf_clusters, traffic_tfidf_clusters, cause_bert_clusters, traffic_bert_clusters = textembedding.predict(X_train)
X_train['Bert_cause'] = cause_tfidf_clusters
X_train['Bert_traffic'] = traffic_tfidf_clusters
X_train['Tfidf_cause'] = cause_bert_clusters
X_train['Tfidf_traffic'] = traffic_bert_clusters

cause_tfidf_clusters, traffic_tfidf_clusters, cause_bert_clusters, traffic_bert_clusters = textembedding.predict(X_test)
X_test['Bert_cause'] = cause_tfidf_clusters
X_test['Bert_traffic'] = traffic_tfidf_clusters
X_test['Tfidf_cause'] = cause_bert_clusters
X_test['Tfidf_traffic'] = traffic_bert_clusters



In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
labels_in_data = sorted(np.unique(y_test))

# parameter thoice and models
models = {
    'Gradient Boosting': (GradientBoostingClassifier(), {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [3, 5],
    }),
    'Random Forest': (RandomForestClassifier(), {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 10],
    }),
    'SVM': (SVC(), {
        'clf__C': [1, 10],
        'clf__kernel': ['linear', 'rbf'],
    }),
    'KNN': (KNeighborsClassifier(), {
        'clf__n_neighbors': [3, 5, 7],
    }),
}

# save results
results = {}

# Train each model with GridSearchCV
for name, (model, param_grid) in models.items():
    pipe = Pipeline([
        ('clf', model)
    ])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    y_pred = grid.best_estimator_.predict(X_test)
    results[name] = {
        'Best Parameters': grid.best_params_,
        'Test Accuracy': grid.best_estimator_.score(X_test, y_test),
        'Classification Report': classification_report(
    y_test, y_pred,
    labels=labels_in_data,
    target_names = [target_mapping[i] for i in labels_in_data if i in target_mapping]
)
    }

# Print results
for model_name, metrics in results.items():
    print(f"\n=== {model_name} ===")
    print("Best Parameters:", metrics['Best Parameters'])
    print("Test Accuracy:", metrics['Test Accuracy'])
    print("Classification Report:\n", metrics['Classification Report'])


=== Gradient Boosting ===
Best Parameters: {'clf__max_depth': 3, 'clf__n_estimators': 200}
Test Accuracy: 0.8675213675213675
Classification Report:
                         precision    recall  f1-score   support

                    触碰       0.31      0.45      0.37        11
                    触礁       0.71      0.33      0.45        15
                others       0.94      0.89      0.92        37
self-sinking (sinking)       0.74      0.89      0.81        36
             Grounding       1.00      1.00      1.00         6
        Fire/explosion       1.00      0.91      0.95        11
         Wind Accident       1.00      0.57      0.73         7
             collision       0.96      0.97      0.96       111

              accuracy                           0.87       234
             macro avg       0.83      0.75      0.77       234
          weighted avg       0.88      0.87      0.87       234


=== Random Forest ===
Best Parameters: {'clf__max_depth': 10, 'clf__n_estimato